# 03 - Evaluate Trained Model

This notebook evaluates the trained checkpoint on test data and selects thresholds.

- Input: `test.csv` manifest and model checkpoint from notebook 02
- Output: metrics table, selected threshold, evaluation report JSON


In [ ]:
# !pip install -q -r /kaggle/working/Face_Anti_Spoofing_Biometric/requirements-kaggle.txt

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/kaggle/working/Face_Anti_Spoofing_Biometric')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fas.evaluation import evaluate_binary_predictions
from fas.train_torch import run_checkpoint_inference

In [ ]:
PREPARED_ROOT = Path('/kaggle/working/celeba_spoof_prepared')
TRAINING_ROOT = Path('/kaggle/working/celeba_spoof_training')
EVAL_ROOT = Path('/kaggle/working/celeba_spoof_eval')
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

manifest_test = PREPARED_ROOT / 'manifests' / 'test.csv'
checkpoint = TRAINING_ROOT / 'best_model.pt'

print('Test manifest exists:', manifest_test.exists())
print('Checkpoint exists:', checkpoint.exists())

In [ ]:
pred_df = run_checkpoint_inference(
    checkpoint_path=str(checkpoint),
    manifest_path=str(manifest_test),
    image_size=80,
)

print('Predictions rows:', len(pred_df))
display(pred_df.head(5))

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
rows = []

for threshold in thresholds:
    metrics = evaluate_binary_predictions(
        live_scores=pred_df['live_score'].tolist(),
        labels=pred_df['label'].tolist(),
        threshold=float(threshold),
    )
    rows.append(
        {
            'threshold': float(threshold),
            'accuracy': metrics.accuracy,
            'apcer': metrics.apcer,
            'bpcer': metrics.bpcer,
            'acer': metrics.acer,
            'tp': metrics.tp,
            'fp': metrics.fp,
            'tn': metrics.tn,
            'fn': metrics.fn,
        }
    )

metrics_df = pd.DataFrame(rows).sort_values('acer').reset_index(drop=True)
display(metrics_df.head(10))


In [ ]:
best_row = metrics_df.iloc[0].to_dict()
print('Best threshold by ACER:')
print(best_row)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(metrics_df['threshold'], metrics_df['acer'], marker='o', label='ACER')
plt.plot(metrics_df['threshold'], metrics_df['accuracy'], marker='s', label='Accuracy')
plt.xlabel('Threshold (live if score >= threshold)')
plt.ylabel('Metric value')
plt.title('Threshold sweep')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
metrics_path = EVAL_ROOT / 'threshold_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

summary_path = EVAL_ROOT / 'best_threshold.json'
summary_path.write_text(json.dumps(best_row, indent=2))

print('Saved threshold metrics:', metrics_path)
print('Saved best threshold summary:', summary_path)